# PTCG Colab trainer — same `v5_s2` command, GPU

Runs `scripts/train_policy.py` on a T4 (or any sm_70+ GPU). Weights export as the same `.npz` `policynet.py` loads.

**Local prep (once):**
```
python -X utf8 scripts/colab/pack.py
```
Upload `out/colab/ptcg_colab.zip` to Drive. Runtime → Change runtime type → **T4 GPU**.

Default job (`td_anchor`): 236k-row Ogerpon TD subset as `--ds`, `pds_all` as `--anchor-ds`, `--advantage-col 0.5`, init `policy_v5_s2_all`. Full `pds_oger_td` is in the zip if you have High-RAM; swap `--ds` if the load fits.

The zip contains the licensed `cg` engine. Keep the Drive file private.

In [ ]:
# ── config ────────────────────────────────────────────────────────────────
ZIP_PATH = "/content/drive/MyDrive/ptcg_colab.zip"   # change if you put it elsewhere
WORK     = "/content/ptcg"

# td_anchor | v5_all | custom
JOB = "td_anchor"

CUSTOM_CMD = None  # if JOB == "custom": list of extra argv after train_policy.py

In [ ]:
from pathlib import Path
import os, sys, zipfile, shutil, subprocess

from google.colab import drive, files
drive.mount("/content/drive")

z = Path(ZIP_PATH)
assert z.exists(), f"zip not found: {z} — upload out/colab/ptcg_colab.zip to Drive"
work = Path(WORK)
if work.exists():
    shutil.rmtree(work)
work.mkdir(parents=True)
with zipfile.ZipFile(z) as zf:
    zf.extractall(work)
os.chdir(work)
print("cwd", os.getcwd())
print("out/", sorted(p.name for p in Path("out").glob("*.npz")))
print("artifacts", [p.name for p in Path("artifacts").iterdir()] if Path("artifacts").exists() else None)

In [ ]:
import torch

device = "cuda"
if not torch.cuda.is_available():
    print("no CUDA — using cpu")
    device = "cpu"
else:
    cap = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    print(f"{name}  compute {cap[0]}.{cap[1]}")
    # Colab T4 is 7.5. Old P100 (6.0) breaks current torch wheels.
    if cap[0] < 7:
        print("GPU too old for this torch; falling back to cpu")
        device = "cpu"
print("device", device)

In [ ]:
import sys, subprocess

ARCH = [
    "--opt-cols", "37", "--state-h", "512,256", "--head-h", "256,128",
    "--pool", "--loss", "listwise",
]

JOBS = {
    # The experiment local RAM blocked: TD AWR + BC corpus leash.
    "td_anchor": [
        "--ds", "artifacts/pds_oger_td_train",
        "--anchor-ds", "artifacts/pds_all",
        "--advantage-col", "0.5",
        "--init", "out/policy_v5_s2_all.npz",
        "--epochs", "5", "--lr", "0.001", "--export-last",
        "--out", "out/policy_v5_s2_oger_td_anchor.npz",
    ],
    # Exact v5_s2 command on the mixed BC corpus already in the zip.
    "v5_all": [
        "--ds", "artifacts/pds_all",
        "--epochs", "12", "--bs", "1024",
        "--init", "out/policy_v5_s2_all.npz",
        "--export-last",
        "--out", "out/policy_v5_s2_all_colab.npz",
        "--seed", "2",
    ],
}

if JOB == "custom":
    extra = CUSTOM_CMD or [_ for _ in ()]
    assert extra, "set CUSTOM_CMD"
else:
    extra = JOBS[JOB]

cmd = [sys.executable, "-X", "utf8", "scripts/train_policy.py", *ARCH,
       "--device", device, *extra]
print("$", " ".join(cmd), flush=True)
r = subprocess.run(cmd, cwd=str(work))
print("exit", r.returncode)
assert r.returncode == 0

In [ ]:
from pathlib import Path
import shutil

out_name = extra[extra.index("--out") + 1]
src = work / out_name
assert src.exists(), src
print(src, f"{src.stat().st_size/1e6:.2f} MB")

# copy to Drive next to the zip
dest = Path(ZIP_PATH).parent / src.name
shutil.copy2(src, dest)
print("Drive ->", dest)
files.download(str(src))